# PRACTICE FILE — Encoding & Scikit-learn Pipelines
### Dataset: IBM Telco Customer Churn

**Rules of this notebook:**
- Every section has a **short brief** (3-5 lines) reminding you of the concept. That is NOT the lesson — you already have the full guide (`Encoding_Pipelines_Guide.docx`) and the worked example (`Telco_Churn_Encoding_Pipelines.ipynb`). Go back to those if a brief isn't enough.
- Every brief is followed by **Tasks**. You write the code in the empty cell below each task. Cells marked `# TODO` are yours to complete — do not skip them, do not peek at the worked-example notebook while attempting a task for the first time.
- Tasks marked **(Challenge)** are harder and optional-but-expected if you want to actually be good at this.
- Some cells contain a **Self-check** — run it after your task. If it prints an error, your code is wrong. Fix it before moving on. A self-check passing does not guarantee a perfect answer, but a failing one guarantees a broken one.
- No solutions are provided in this file. That is the point of a practice file.

**Setup below is given to you — do not skip running it.**


## 0. Setup (Given — just run these)

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

print("Setup complete.")

Setup complete.


In [2]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
df = df.drop(columns=['customerID'])

print("Shape:", df.shape)
df.head()

Shape: (7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Encoding — Brief

Machine learning models only understand numbers. Encoding converts
categorical (text) columns into numeric form. **Nominal** data (no
order, e.g. `gender`) usually gets **One-Hot Encoding**. **Ordinal**
data (real order, e.g. `Contract`) gets **Ordinal Encoding** with a
manually defined order. High-cardinality columns often use
**Frequency** or **Target Encoding**. Label Encoding is best kept for
the target column or tree-based models.


### Task 1.1 — Clean `TotalCharges`
`TotalCharges` is loaded as text (`object`) because a few rows contain
blank strings instead of numbers.
- Convert `TotalCharges` to a numeric dtype.
- Print how many missing values exist after the conversion.
- Do **not** drop or fill them yet — that comes later with a proper `SimpleImputer`.


In [3]:
# df.info()
df.TotalCharges.isna().sum()

np.int64(0)

In [4]:
# TODO: convert df['TotalCharges'] to numeric, coercing errors to NaN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
# TODO: print the number of missing values in TotalCharges after conversion
print(df.TotalCharges.isna().sum())
df[df.TotalCharges.isna()]


11


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,No,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,Male,0,No,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,Yes,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
3331,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
3826,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
4380,Female,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5218,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6670,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,Yes,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


### Task 1.2 — Label Encode the target column
Encode `Churn` ('Yes'/'No') into a numeric column called `Churn_encoded`
using `LabelEncoder`. Print which string mapped to 0 and which to 1.


In [5]:
# TODO: create Churn_encoded using LabelEncoder
df.Churn.unique()
le = LabelEncoder()
df["Churn_encoded"] = le.fit_transform(df.Churn)


In [6]:
df[["Churn", "Churn_encoded"]]

,Churn,Churn_encoded
0,No,0
1,No,0
2,Yes,1
3,No,0
4,Yes,1
...,...,...
7038,No,0
7039,No,0
7040,No,0
7041,Yes,1


In [7]:
# TODO: print the mapping (which class -> which integer)
print(dict(zip(le.classes_, le.transform(le.classes_))))
df.head()

{'No': np.int64(0), 'Yes': np.int64(1)}


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Churn_encoded
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


### Task 1.3 — Train/Test split BEFORE further encoding
Split the data into `X_train`, `X_test`, `y_train`, `y_test`
(80/20, `random_state=42`, `stratify` on the target). Use `Churn_encoded`
as `y`, and drop both `Churn` and `Churn_encoded` from `X`.

**Why does the split have to happen before you fit any Frequency/Target
encoder?** Answer this in a one-line comment in your code cell.


In [8]:
# Build X and y
X = df.drop(columns=["Churn", "Churn_encoded"])
y = df["Churn_encoded"]

# Split: 80 percent train, 20 percent test, same class balance in both
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Why split first: encoders that learn from data (frequency, target) must only see
# the training rows. If they see the test rows too, test information leaks into training.
print(X_train.shape, X_test.shape)

(5634, 19) (1409, 19)


### Self-check 1.3

In [9]:
# Self-check — do not edit, just run
try:
    assert X_train.shape[0] == 5634 and X_test.shape[0] == 1409
    assert 'Churn' not in X_train.columns and 'Churn_encoded' not in X_train.columns
    print("Self-check passed.")
except NameError:
    print("X_train / X_test not defined yet — complete Task 1.3 first.")
except AssertionError:
    print("Shapes or columns look wrong — check your split and dropped columns.")

Self-check passed.


### Task 1.4 — One-Hot Encode two nominal columns
Using `OneHotEncoder` (with `drop='first'` and `handle_unknown='ignore'`),
one-hot encode the `gender` and `InternetService` columns of `X_train`.
Print the resulting shape and the generated column names
(`get_feature_names_out`).


In [10]:
ohe = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
ohe_out = ohe.fit_transform(X_train[["gender", "InternetService"]])

print("Output shape:", ohe_out.shape)
print("Feature names:", ohe.get_feature_names_out())

Output shape: (5634, 3)
Feature names: ['gender_Male' 'InternetService_Fiber optic' 'InternetService_No']


### Task 1.5 — Ordinal Encode `Contract`
`Contract` has a real order: `Month-to-month` < `One year` < `Two year`.
Use `OrdinalEncoder` with a manually specified `categories` order (do
**not** let it guess alphabetically) to create `Contract_ordinal`.
Print the unique (`Contract`, `Contract_ordinal`) pairs sorted by the
encoded value, to prove the order is correct.


In [11]:
contract_order = [["Month-to-month", "One year", "Two year"]]
ord_enc = OrdinalEncoder(categories=contract_order)

X_train_ord = X_train.copy()          # work on a copy, keep X_train unchanged
X_train_ord["Contract_ordinal"] = ord_enc.fit_transform(X_train_ord[["Contract"]])

pairs = (
    X_train_ord[["Contract", "Contract_ordinal"]]
    .drop_duplicates()
    .sort_values("Contract_ordinal")
)
print(pairs)

            Contract  Contract_ordinal
3738  Month-to-month               0.0
2257        One year               1.0
4860        Two year               2.0


### Task 1.6 — Frequency Encode `PaymentMethod`
Implement Frequency Encoding **manually with pandas** (no sklearn class
for this one). Use only `X_train` to compute the frequency map — this
matters, think about why. Apply the same map to a copy of `X_train`
as a new column `PaymentMethod_freq`.


In [12]:
# The map is learned from X_train only
freq_map = X_train["PaymentMethod"].value_counts(normalize=True)

X_train_freq = X_train.copy()
X_train_freq["PaymentMethod_freq"] = X_train_freq["PaymentMethod"].map(freq_map)

print(freq_map.sort_values(ascending=False))
print(X_train_freq[["PaymentMethod", "PaymentMethod_freq"]].head())

# Why train only: the test set is "new data". Its counts must not shape the encoder.
# A category that is missing from the train map would become NaN in the test set.

PaymentMethod
Electronic check             0.335641
Mailed check                 0.228257
Bank transfer (automatic)    0.220802
Credit card (automatic)      0.215300
Name: proportion, dtype: float64
                PaymentMethod  PaymentMethod_freq
3738         Electronic check            0.335641
3151             Mailed check            0.228257
4860             Mailed check            0.228257
3867  Credit card (automatic)            0.215300
3810         Electronic check            0.335641


### Task 1.7 (Challenge) — Target Encode `PaymentMethod`
Compute the mean of `y_train` grouped by `PaymentMethod`, using
`X_train`/`y_train` ONLY. Create `PaymentMethod_target` on a copy of
`X_train`. Then explain in a comment: what would go wrong if you
computed this mean using the FULL dataset (train + test) instead?


In [13]:
train_df = X_train.copy()
train_df["y"] = y_train

target_map = train_df.groupby("PaymentMethod")["y"].mean()

X_train_target = X_train.copy()
X_train_target["PaymentMethod_target"] = X_train_target["PaymentMethod"].map(target_map)

print(target_map.sort_values(ascending=False))

# Leakage risk: the full dataset includes the test labels. The encoded value would then
# carry information about test answers, so test accuracy would look better than it
# really is on new data.

PaymentMethod
Electronic check             0.457430
Mailed check                 0.192846
Bank transfer (automatic)    0.161576
Credit card (automatic)      0.149217
Name: y, dtype: float64


## 2. ColumnTransformer — Brief

Real datasets mix numeric and categorical columns that need different
treatment. `ColumnTransformer` routes each group of columns to its own
transformer (e.g. `StandardScaler` for numeric, `OneHotEncoder` for
categorical) and concatenates the results into one feature matrix — in
a single object, safe to reuse identically on train, test, and
production data.


### Task 2.1 — Define your column groups
Create two Python lists:
- `numeric_features`: all numeric columns (`tenure`, `MonthlyCharges`, `TotalCharges`, `SeniorCitizen`)
- `categorical_features`: every remaining column in `X_train` that is NOT in `numeric_features`

Do not hardcode the categorical list by hand — derive it programmatically
from `X_train.columns`.


In [14]:
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
categorical_features = [c for c in X_train.columns if c not in numeric_features]

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

Numeric: ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
Categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


### Task 2.2 — Build the ColumnTransformer
Build a `ColumnTransformer` named `preprocessor` with:
- a numeric branch: `SimpleImputer(strategy='median')` -> `StandardScaler()`
- a categorical branch: `SimpleImputer(strategy='most_frequent')` -> `OneHotEncoder(handle_unknown='ignore')`

Use `Pipeline` for each branch (two mini-pipelines inside the
ColumnTransformer — this is a very common real-world pattern).


In [15]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

### Task 2.3 — Fit and inspect
`fit_transform` your `preprocessor` on `X_train`. Print the shape before
and after transformation, and explain in one comment why the number of
columns changed.


In [16]:
X_train_transformed = preprocessor.fit_transform(X_train)

print("Before:", X_train.shape)
print("After: ", X_train_transformed.shape)

# The column count grew because One-Hot Encoding turns each categorical column into
# one new column per category (for example, Contract becomes 3 columns).

Before: (5634, 19)
After:  (5634, 45)


### Self-check 2.3

In [17]:
try:
    assert X_train_transformed.shape[0] == X_train.shape[0]
    assert X_train_transformed.shape[1] > X_train.shape[1]
    print("Self-check passed.")
except NameError:
    print("preprocessor / X_train_transformed not defined yet — complete Task 2.2-2.3 first.")
except AssertionError:
    print("Shape looks wrong — the transformed matrix should have MORE columns than the raw one (because of One-Hot Encoding).")

Self-check passed.


### Task 2.4 (Challenge) — `remainder` parameter
Rebuild the same `ColumnTransformer` but this time deliberately leave
ONE categorical column out of `categorical_features` when you define
the transformer list. Run it once with `remainder='drop'` and once with
`remainder='passthrough'`. Print the resulting shapes for both and
explain the difference in one comment.


In [18]:
def build_preprocessor(remainder):
    # PaperlessBilling is left out of the categorical list on purpose
    cat_partial = [c for c in categorical_features if c != "PaperlessBilling"]
    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, cat_partial),
        ],
        remainder=remainder,
    )

pre_drop = build_preprocessor("drop")
pre_pass = build_preprocessor("passthrough")

print("remainder='drop':       ", pre_drop.fit_transform(X_train).shape)
print("remainder='passthrough':", pre_pass.fit_transform(X_train).shape)

# 'drop' removes the left-out column from the output. 'passthrough' keeps it as it is,
# so the output has one more column. That column stays as raw text (not encoded),
# so most models would fail on it. Use passthrough only for columns that are already numeric.

remainder='drop':        (5634, 43)


remainder='passthrough': (5634, 44)


## 3. Pipeline — Brief

A `Pipeline` chains preprocessing steps and a final estimator into ONE
object with a single `.fit()` / `.predict()`. This prevents data
leakage, makes cross-validation correct, and is what you'd actually
ship to production.


### Task 3.1 — Build a full Pipeline
Combine your `preprocessor` (from Task 2.2, using `remainder='drop'`
version) with a `LogisticRegression(max_iter=1000, random_state=42)`
into a `Pipeline` named `log_reg_pipeline`.


In [19]:
log_reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])
log_reg_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

### Task 3.2 — Fit, predict, evaluate
Fit `log_reg_pipeline` on the training data, predict on the test data,
and print the accuracy and full `classification_report`.


In [20]:
log_reg_pipeline.fit(X_train, y_train)
y_pred = log_reg_pipeline.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred))

Accuracy: 0.8055
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



### Self-check 3.2

In [21]:
try:
    acc = accuracy_score(y_test, log_reg_pipeline.predict(X_test))
    assert acc > 0.70
    print(f"Self-check passed. Accuracy = {acc:.4f}")
except NameError:
    print("log_reg_pipeline not fitted yet — complete Task 3.1-3.2 first.")
except AssertionError:
    print(f"Accuracy ({acc:.4f}) looks too low — double check your preprocessing and target.")

Self-check passed. Accuracy = 0.8055


### Task 3.3 — Swap the model
Copy your pipeline and swap only the final estimator for a
`RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42)`.
You should only need to change ONE line. Fit and print its test accuracy.


In [22]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42)),
])

rf_pipeline.fit(X_train, y_train)
print("Random Forest accuracy:", round(accuracy_score(y_test, rf_pipeline.predict(X_test)), 4))

Random Forest accuracy: 0.8027


### Task 3.4 (Challenge) — GridSearchCV
Use `GridSearchCV` on `log_reg_pipeline` to search over at least 4
different values of `classifier__C`, with `cv=5`. Print the best
parameters and the best cross-validated accuracy.


In [23]:
param_grid = {"classifier__C": [0.01, 0.1, 1, 10, 100]}

grid_search = GridSearchCV(log_reg_pipeline, param_grid, cv=5, scoring="accuracy")
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV accuracy:", round(grid_search.best_score_, 4))

Best parameters: {'classifier__C': 100}
Best CV accuracy: 0.8048


## 4. make_pipeline vs Pipeline — Brief

`make_pipeline()` builds the exact same kind of object as `Pipeline()`,
but auto-generates step names from each object's lowercase class name
instead of letting you choose them. Behavior is identical; only the
step names (and therefore `GridSearchCV` parameter names) differ.


### Task 4.1 — Rebuild with make_pipeline
Rebuild the exact same Logistic Regression pipeline from Task 3.1, this
time using `make_pipeline()` instead of `Pipeline()`. Call it `pipe_mp`.


In [24]:
pipe_mp = make_pipeline(preprocessor, LogisticRegression(max_iter=1000, random_state=42))
pipe_mp

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the differe

### Task 4.2 — Compare step names
Fit `pipe_mp` on the training data. Print `.named_steps.keys()` for
BOTH `log_reg_pipeline` and `pipe_mp` side by side.


In [25]:
pipe_mp.fit(X_train, y_train)

print("Pipeline:     ", list(log_reg_pipeline.named_steps.keys()))
print("make_pipeline:", list(pipe_mp.named_steps.keys()))

Pipeline:      ['preprocessor', 'classifier']
make_pipeline: ['columntransformer', 'logisticregression']


### Task 4.3 — Verify identical predictions
Prove that `log_reg_pipeline` and `pipe_mp` produce IDENTICAL
predictions on `X_test` using `np.array_equal`.


In [26]:
same = np.array_equal(log_reg_pipeline.predict(X_test), pipe_mp.predict(X_test))
print("Predictions identical:", same)

Predictions identical: True


### Self-check 4.3

In [27]:
try:
    same = np.array_equal(log_reg_pipeline.predict(X_test), pipe_mp.predict(X_test))
    assert same
    print("Self-check passed — predictions are identical, as expected.")
except NameError:
    print("pipe_mp not fitted yet — complete Task 4.1-4.2 first.")
except AssertionError:
    print("Predictions differ — something in pipe_mp is not built the same way as log_reg_pipeline.")

Self-check passed — predictions are identical, as expected.


### Task 4.4 (Challenge) — GridSearchCV with make_pipeline
Repeat Task 3.4's grid search, but this time on `pipe_mp`. You will need
to figure out the correct auto-generated parameter name yourself
(hint: print `pipe_mp.get_params().keys()` and look for the one ending
in `__C`).


In [28]:
# The step name is the lowercase class name, so the C parameter is "logisticregression__C"
print([k for k in pipe_mp.get_params().keys() if k.endswith("__C")])

param_grid_mp = {"logisticregression__C": [0.01, 0.1, 1, 10, 100]}

grid_mp = GridSearchCV(pipe_mp, param_grid_mp, cv=5, scoring="accuracy")
grid_mp.fit(X_train, y_train)

print("Best parameters:", grid_mp.best_params_)
print("Best CV accuracy:", round(grid_mp.best_score_, 4))

['logisticregression__C']


Best parameters: {'logisticregression__C': 100}
Best CV accuracy: 0.8048


## 5. Final Mini-Project (Challenge)

Put everything together. No more step-by-step hand-holding.

**Your task:**
1. Build the best pipeline you can for predicting `Churn` on this
   dataset (you choose: Logistic Regression, Random Forest, or anything
   else from sklearn you're comfortable with).
2. Tune at least one hyperparameter with `GridSearchCV` or
   `cross_val_score`.
3. Report final **test set** accuracy AND a full `classification_report`.
4. Save your final trained pipeline to a file called
   `my_churn_pipeline.pkl` using `joblib`.
5. In a markdown cell below your code, write 3-4 sentences explaining:
   - Which encoding choices you made and why.
   - Whether `ColumnTransformer` + `Pipeline` made this easier than doing
     it manually, and specifically how.


In [29]:
import joblib

# One pipeline, two candidate models. GridSearchCV picks the best one using cross-validation
final_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

param_grid_final = [
    {
        "classifier": [LogisticRegression(max_iter=1000, random_state=42)],
        "classifier__C": [0.01, 0.1, 1, 10],
    },
    {
        "classifier": [RandomForestClassifier(random_state=42)],
        "classifier__n_estimators": [200, 400],
        "classifier__max_depth": [6, 10],
    },
]

final_search = GridSearchCV(final_pipe, param_grid_final, cv=5, scoring="accuracy", n_jobs=-1)
final_search.fit(X_train, y_train)

print("Best parameters:", final_search.best_params_)
print("Best CV accuracy:", round(final_search.best_score_, 4))

# Final check on the test set, used only once
best_model = final_search.best_estimator_
y_pred_final = best_model.predict(X_test)
print("Test accuracy:", round(accuracy_score(y_test, y_pred_final), 4))
print(classification_report(y_test, y_pred_final))

# Save the trained pipeline, then load it back to confirm it works
joblib.dump(best_model, "my_churn_pipeline.pkl")
loaded = joblib.load("my_churn_pipeline.pkl")
print("Reloaded pipeline gives the same predictions:",
      np.array_equal(loaded.predict(X_test), y_pred_final))

Best parameters: {'classifier': LogisticRegression(max_iter=1000, random_state=42), 'classifier__C': 10}
Best CV accuracy: 0.8046
Test accuracy: 0.8055
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

Reloaded pipeline gives the same predictions: True


### Your write-up

I used One-Hot Encoding for the text columns, because they have no real order and it avoids giving the model a fake ranking. Numeric columns got median imputation and scaling, which keeps missing values from breaking the model and helps Logistic Regression train well. ColumnTransformer and Pipeline made this much easier than doing it by hand: every step is learned on the training data only, so there is no leakage, and the same object can be reused on test data and saved as one file. It also let GridSearchCV tune the model while redoing the preprocessing correctly inside each fold.

---
### Done?
Go back to `Encoding_Pipelines_Guide.docx` Section 5 (Cheat Sheet) and
`Telco_Churn_Encoding_Pipelines.ipynb` and compare your answers. If
something doesn't match and you don't understand why — that's the part
you actually need to study, not the part you got right.
